# Sentiment Analysis of Product Reviews
### Classifying customer reviews as Positive, Neutral, or Negative using NLP and Machine Learning

This notebook demonstrates the step-by-step pipeline for building a sentiment classification model using the **Naive Bayes** algorithm and **TF-IDF** features.

## Step 1: Import Dependencies

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import joblib
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Ensure NLTK resources are downloaded
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

## Step 2: Load the Dataset

In [ ]:
dataset_path = '../dataset/reviews.csv'
if not os.path.exists(dataset_path):
    # Fallback to local directory if run from notebook directory
    dataset_path = 'dataset/reviews.csv'
    if not os.path.exists(dataset_path):
        dataset_path = 'reviews.csv'

df = pd.read_csv(dataset_path)
print(f"Loaded dataset with {len(df)} reviews.")
print("\nFirst 5 rows:")
df.head()

## Step 3: Explore Sentiment Distribution

In [ ]:
df['Sentiment'].value_counts()

## Step 4: Text Preprocessing
The text preprocessing pipeline contains:
1. Conversion to lowercase.
2. Removal of punctuation and unnecessary characters (numbers, tags).
3. Tokenization.
4. Removal of English stopwords.

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove HTML tags & non-alphabetic chars
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords & single-letter words
    cleaned_tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(cleaned_tokens)

# Apply preprocessing
df['Cleaned_Text'] = df['Review_Text'].apply(preprocess_text)
print("Sample of preprocessed text:")
df[['Review_Text', 'Cleaned_Text']].head()

## Step 5: Split Data into Train and Test Sets

In [ ]:
X = df['Cleaned_Text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## Step 6: Text Vectorization using TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(max_features=2500, min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF training matrix shape: {X_train_tfidf.shape}")

## Step 7: Train the Naive Bayes Classifier

In [ ]:
model = MultinomialNB(alpha=1.0)
model.fit(X_train_tfidf, y_train)
print("Multinomial Naive Bayes model trained successfully.")

## Step 8: Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
labels = ['Negative', 'Neutral', 'Positive']
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df

## Step 9: Save the Model and Vectorizer

In [ ]:
# Ensure model directory exists
os.makedirs('../model', exist_ok=True)
os.makedirs('model', exist_ok=True)

# Try saving to standard path
try:
    joblib.dump(model, '../model/sentiment_model.pkl')
    joblib.dump(vectorizer, '../model/tfidf_vectorizer.pkl')
    print("Model and Vectorizer saved successfully to '../model/'.")
except Exception:
    joblib.dump(model, 'model/sentiment_model.pkl')
    joblib.dump(vectorizer, 'model/tfidf_vectorizer.pkl')
    print("Model and Vectorizer saved successfully to 'model/'.")